# Stage 5 — Confirmation & report

**Pipeline stage 5 of 5**: patient-level split → phase segmentation → train + explain →
candidate discovery → **confirm & report**.

This is the moment the whole pipeline was built around. Every recording from every patient in
the **confirmation cohort** has sat untouched since Stage 1 — never trained on, never used to
tune anything, never looked at while searching for candidates. This notebook opens that locked
drawer exactly once.

## What happens here, and nothing more

For each candidate marker that survived Stage 4 (statistically significant, fold-stable,
occlusion-validated on the *development* set), the **same statistical test is run again,
unchanged**, on the confirmation cohort:

- **No re-searching.** Only the specific (phase, frequency-band, site) cells already
  shortlisted in Stage 4 are tested — not a new 256-cell scan. Searching again here would just
  be Stage 4 with extra steps, and would need its own confirmation cohort in turn.
- **No re-tuning.** No model retraining, no threshold adjustment, no picking a "better"
  frequency band. The candidate definition is frozen as of Stage 4.
- **Direction matters as much as significance.** A candidate only counts as confirmed if the
  confirmation-set result is *both* statistically significant *and* points the same direction
  (MI higher or lower than Normal, matching development) — significant-but-flipped is not a
  replication, and this notebook checks for that explicitly rather than only checking p-values.

## Ensemble inference, not out-of-fold

Stage 3's out-of-fold trick doesn't apply here — every confirmation patient is equally unseen
by *all* `N_FOLDS` models (none of them ever trained on it), so instead of picking one fold's
model, every confirmation recording is explained by the **average** of all fold models' Grad-CAM
heatmaps and predictions. This also means confirmation-set recordings need segmenting for the
first time (Stage 2 only ever touched development-set recordings) — using the exact same
`pipeline_common.segmentation` code, unchanged.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ---- Setup ----------------------------------------------------------------
import sys
import json
import datetime
import pathlib

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

package_path = "/content/drive/MyDrive/Research 2026/mi_marker_pipeline"
if package_path not in sys.path:
    sys.path.insert(0, package_path)

from pipeline_common import config
from pipeline_common.audio import load_and_resample, pad_or_trim
from pipeline_common.features import calculate_features, n_freq_bins
from pipeline_common.models import make_model
from pipeline_common.segmentation import segment_recording_heuristic
from pipeline_common.explain import compute_ensemble_gradcam
from pipeline_common.discovery import (
    generate_frequency_bands, compute_recording_energy_table, aggregate_to_patient_level,
    confirm_candidates,
)

import random
random.seed(config.SEED); np.random.seed(config.SEED); torch.manual_seed(config.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SPLIT_DIR = config.ARTIFACTS_DIR / "01_split"
TRAIN_DIR = config.ARTIFACTS_DIR / "03_train_explain"
DISCOVERY_DIR = config.ARTIFACTS_DIR / "04_candidate_discovery"
STAGE_DIR = config.ARTIFACTS_DIR / "05_confirmation"
STAGE_DIR.mkdir(parents=True, exist_ok=True)

print("Using device:", DEVICE)
print("Saving this stage's output to:", STAGE_DIR)

Using device: cuda
Saving this stage's output to: /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation


## 1. Load the confirmation cohort + Stage 4's shortlist

If Stage 4 found no occlusion-validated candidates, there's nothing to confirm -- this stops
here rather than fabricating a report.

In [3]:
metadata = pd.read_csv(SPLIT_DIR / "metadata_with_split.csv")
confirmation_metadata = metadata[metadata["split"] == "confirmation"].reset_index(drop=True)

candidates_df = pd.read_csv(DISCOVERY_DIR / "candidate_markers_final.csv")

print(f"{len(confirmation_metadata)} confirmation-set recordings "
      f"({confirmation_metadata['patient_id'].nunique()} patients) -- untouched until now")
print(f"{len(candidates_df)} candidate(s) from Stage 4 to confirm")

if len(candidates_df) == 0:
    print("\nNo candidates to confirm -- Stage 4 didn't find anything that survived statistical "
          "testing, fold stability, AND occlusion validation on the development set. That's a "
          "legitimate, reportable outcome: either add more development data and revisit Stage "
          "4, or report that no acoustic marker was found under this methodology.")
    raise SystemExit("Nothing to confirm -- see message above.")

candidates_df

672 confirmation-set recordings (28 patients) -- untouched until now
2 candidate(s) from Stage 4 to confirm


,site,phase_label,freq_low,freq_high,n_mi,n_normal,median_mi,median_normal,cliffs_delta,p_value,...,fold_agree,fold_total,fold_stability_frac,n_recordings_tested,baseline_accuracy,occluded_accuracy,occlusion_effect,random_control_effects,percentile_vs_random,occlusion_validated
0,lusb,S2,200.0,300.0,56,56,0.141395,0.204385,-0.480867,0.000012,...,5.0,5.0,1.0,672,0.892857,0.870536,0.022321,"[0.0, 0.013392857142857206, 0.0, 0.01190476190...",1.0,True
1,apex,S1,100.0,200.0,55,56,0.156684,0.224714,-0.401299,0.000270,...,5.0,5.0,1.0,663,0.880845,0.847662,0.033183,"[0.006033182503770718, 0.018099547511312153, 0...",1.0,True


## 2. Segment the confirmation set

First time these recordings are touched by anything -- same heuristic segmenter as Stage 2,
unchanged.

In [4]:
confirm_boundary_rows = []
confirm_quality_rows = []

for i, row in confirmation_metadata.iterrows():
    signal = pad_or_trim(load_and_resample(row["filepath"], config.SAMPLE_RATE),
                         config.SAMPLE_RATE, config.CLIP_SECONDS)
    boundaries, quality = segment_recording_heuristic(signal, config.SAMPLE_RATE)
    for b in boundaries:
        confirm_boundary_rows.append({
            "filepath": row["filepath"], "patient_id": row["patient_id"],
            "class_name": row["class_name"], "site": row["site"], "segment": row["segment"], **b,
        })
    confirm_quality_rows.append({
        "filepath": row["filepath"], "patient_id": row["patient_id"],
        "class_name": row["class_name"], "site": row["site"], "segment": row["segment"], **quality,
    })
    if (i + 1) % 100 == 0 or (i + 1) == len(confirmation_metadata):
        print(f"  segmented {i + 1}/{len(confirmation_metadata)} recordings")

confirm_boundaries_df = pd.DataFrame(confirm_boundary_rows)
confirm_quality_df = pd.DataFrame(confirm_quality_rows)
print(f"\n{confirm_quality_df['quality_flag'].value_counts().to_dict()}")

  segmented 100/672 recordings
  segmented 200/672 recordings
  segmented 300/672 recordings
  segmented 400/672 recordings
  segmented 500/672 recordings
  segmented 600/672 recordings
  segmented 672/672 recordings

{'ok': 618, 'low_confidence_irregular_cycles': 49, 'low_confidence_hr_out_of_range': 4, 'failed_too_few_peaks': 1}


## 3. Ensemble Grad-CAM on every confirmation recording

Every fold's model, averaged -- see the intro for why this replaces Stage 3's out-of-fold
scheme here.

In [5]:
N_FREQ_BINS = n_freq_bins()
fold_models = []
for k in range(config.N_FOLDS):
    m = make_model(N_FREQ_BINS, DEVICE)
    m.load_state_dict(torch.load(TRAIN_DIR / "models" / f"fold_{k}.pt", map_location=DEVICE))
    m.eval()
    fold_models.append(m)
print(f"Loaded {len(fold_models)} fold models for ensemble inference")

confirm_pred_rows = []
confirm_cams = []
for i, row in confirmation_metadata.iterrows():
    signal = pad_or_trim(load_and_resample(row["filepath"], config.SAMPLE_RATE),
                         config.SAMPLE_RATE, config.CLIP_SECONDS)
    feat = torch.from_numpy(calculate_features(signal)).float().unsqueeze(0).to(DEVICE)
    cam, probs = compute_ensemble_gradcam(fold_models, feat, class_idx=config.GRADCAM_TARGET_CLASS)
    confirm_cams.append(cam)
    confirm_pred_rows.append({
        "filepath": row["filepath"], "patient_id": row["patient_id"],
        "class_name": row["class_name"], "site": row["site"], "segment": row["segment"],
        "true_label": int(row["label"]), "predicted_prob_mi": float(probs[1]),
        "predicted_label": int(probs.argmax()),
        "fold_index": -1,  # placeholder -- confirmation patients aren't part of any k-fold
                           # (every model explains them equally, see the ensemble step above),
                           # this only exists so compute_recording_energy_table's schema
                           # (built for Stage 4's real per-fold data) works unchanged here too
    })
    if (i + 1) % 100 == 0 or (i + 1) == len(confirmation_metadata):
        print(f"  explained {i + 1}/{len(confirmation_metadata)} recordings")

confirm_oof_df = pd.DataFrame(confirm_pred_rows)
confirm_cams_array = np.stack(confirm_cams)
assert len(confirm_oof_df) == confirm_cams_array.shape[0]

Loaded 5 fold models for ensemble inference
  explained 100/672 recordings
  explained 200/672 recordings
  explained 300/672 recordings
  explained 400/672 recordings
  explained 500/672 recordings
  explained 600/672 recordings
  explained 672/672 recordings


## 4. Confirmation-set classifier performance

A sanity check alongside the marker-specific claims below: does the classifier even generalise
to these never-seen patients at all?

In [6]:
y_true = confirm_oof_df["true_label"].values
y_pred = confirm_oof_df["predicted_label"].values
y_prob = confirm_oof_df["predicted_prob_mi"].values

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")
specificity = tn / (tn + fp) if (tn + fp) else float("nan")
auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan")

print(f"Confirmation-set accuracy:    {acc:.3f}")
print(f"Confirmation-set sensitivity: {sensitivity:.3f}")
print(f"Confirmation-set specificity: {specificity:.3f}")
print(f"Confirmation-set AUC:         {auc:.3f}")

classifier_performance_confirmation = {"accuracy": float(acc), "sensitivity": float(sensitivity),
                                       "specificity": float(specificity), "auc": float(auc)}

Confirmation-set accuracy:    0.876
Confirmation-set sensitivity: 0.827
Confirmation-set specificity: 0.926
Confirmation-set AUC:         0.942


## 5. Energy table + the confirmatory test

Same binning and aggregation functions as Stage 4, applied to the confirmation set's own
heatmaps and phase boundaries -- then the targeted, pre-registered test against just the
Stage 4 shortlist.

In [7]:
freq_bands = generate_frequency_bands(config.FREQUENCY_HIGH, config.FREQUENCY_BAND_WIDTH_HZ)

confirm_recording_energy_df = compute_recording_energy_table(
    confirm_oof_df, confirm_cams_array, confirm_boundaries_df, freq_bands,
    config.FREQUENCY_HIGH, config.CLIP_SECONDS, aggregation=config.ENERGY_AGGREGATION,
)
confirm_patient_energy_df = aggregate_to_patient_level(confirm_recording_energy_df)

print(f"{len(confirm_recording_energy_df)} recording-level rows -> "
      f"{len(confirm_patient_energy_df)} patient-level rows on the confirmation set")

confirmation_results = confirm_candidates(
    confirm_patient_energy_df, candidates_df,
    min_patients_per_group=config.MIN_PATIENTS_PER_GROUP_FOR_TEST, alpha=config.FDR_ALPHA,
)
confirmation_results

42944 recording-level rows -> 7104 patient-level rows on the confirmation set


,site,phase_label,freq_low,freq_high,dev_median_mi,dev_median_normal,dev_cliffs_delta,dev_p_value,confirm_n_mi,confirm_n_normal,confirm_median_mi,confirm_median_normal,confirm_cliffs_delta,confirm_p_value,direction_agrees,insufficient_data,confirm_fdr_q_value,confirmed
0,lusb,S2,200.0,300.0,0.141395,0.204385,-0.480867,0.000012,14,14,0.124652,0.229430,-0.816327,0.000259,True,False,0.000519,True
1,apex,S1,100.0,200.0,0.156684,0.224714,-0.401299,0.000270,14,14,0.156510,0.221162,-0.744898,0.000865,True,False,0.000865,True


## 6. Final report

In [8]:
confirmed = confirmation_results[confirmation_results["confirmed"]].sort_values("confirm_p_value")

print("=" * 72)
print(f"STAGE 5 CONFIRMATION RESULTS")
print("=" * 72)
print(f"\n{len(confirmed)} of {len(confirmation_results)} candidate(s) confirmed on the "
      f"{confirmation_metadata['patient_id'].nunique()}-patient confirmation cohort.\n")

for _, c in confirmation_results.iterrows():
    status = "CONFIRMED" if c["confirmed"] else "NOT CONFIRMED"
    print(f"\n[{status}] {c['phase_label']} + {c['freq_low']:.0f}-{c['freq_high']:.0f} Hz + {c['site']}")
    if c["insufficient_data"]:
        print(f"    Confirmation set had too few patients in this cell "
              f"(n_mi={c['confirm_n_mi']}, n_normal={c['confirm_n_normal']}) -- inconclusive, not refuted.")
        continue
    print(f"    Development:  median MI={c['dev_median_mi']:.4f}  median Normal={c['dev_median_normal']:.4f}  "
          f"(Cliff's delta {c['dev_cliffs_delta']:.3f}, p={c['dev_p_value']:.4f})")
    print(f"    Confirmation: median MI={c['confirm_median_mi']:.4f}  median Normal={c['confirm_median_normal']:.4f}  "
          f"(Cliff's delta {c['confirm_cliffs_delta']:.3f}, p={c['confirm_p_value']:.4f}, "
          f"q={c['confirm_fdr_q_value']:.4f})")
    print(f"    Direction agrees with development: {c['direction_agrees']}")

if len(confirmed) > 0:
    print(f"\n\nThese {len(confirmed)} marker(s) satisfy every stage of the pipeline: recur "
          f"across development patients, stable across folds, statistically significant with "
          f"FDR correction, causally load-bearing under occlusion, AND replicate -- same "
          f"direction, still significant -- on patients no part of this pipeline has ever "
          f"trained on, tuned against, or searched over. That is the patient-independent claim "
          f"the original research question asked for.")
else:
    print(f"\n\nNone of the development-set candidates replicated on the confirmation cohort. "
          f"This is a legitimate, reportable result, not a failure of the pipeline -- it means "
          f"the candidates found in Stage 4, however promising they looked there, don't (yet) "
          f"hold up as patient-independent markers. Worth reporting alongside sample size: a "
          f"small confirmation cohort has limited power to confirm real but modest effects.")

STAGE 5 CONFIRMATION RESULTS

2 of 2 candidate(s) confirmed on the 28-patient confirmation cohort.


[CONFIRMED] S2 + 200-300 Hz + lusb
    Development:  median MI=0.1414  median Normal=0.2044  (Cliff's delta -0.481, p=0.0000)
    Confirmation: median MI=0.1247  median Normal=0.2294  (Cliff's delta -0.816, p=0.0003, q=0.0005)
    Direction agrees with development: True

[CONFIRMED] S1 + 100-200 Hz + apex
    Development:  median MI=0.1567  median Normal=0.2247  (Cliff's delta -0.401, p=0.0003)
    Confirmation: median MI=0.1565  median Normal=0.2212  (Cliff's delta -0.745, p=0.0009, q=0.0009)
    Direction agrees with development: True


These 2 marker(s) satisfy every stage of the pipeline: recur across development patients, stable across folds, statistically significant with FDR correction, causally load-bearing under occlusion, AND replicate -- same direction, still significant -- on patients no part of this pipeline has ever trained on, tuned against, or searched over. That is the 

## 7. Save this stage's outputs, and a human-readable final report

In [9]:
confirm_boundaries_df.to_csv(STAGE_DIR / "phase_boundaries_confirmation.csv", index=False)
confirm_quality_df.to_csv(STAGE_DIR / "segmentation_quality_confirmation.csv", index=False)
confirm_oof_df.to_csv(STAGE_DIR / "confirmation_predictions.csv", index=False)
np.savez_compressed(STAGE_DIR / "gradcam_heatmaps_confirmation.npz", cams=confirm_cams_array)
confirm_recording_energy_df.to_csv(STAGE_DIR / "recording_energy_table_confirmation.csv", index=False)
confirm_patient_energy_df.to_csv(STAGE_DIR / "patient_energy_table_confirmation.csv", index=False)
confirmation_results.to_csv(STAGE_DIR / "confirmation_results.csv", index=False)

config_used = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "n_confirmation_recordings": int(len(confirmation_metadata)),
    "n_confirmation_patients": int(confirmation_metadata["patient_id"].nunique()),
    "n_candidates_tested": int(len(candidates_df)),
    "n_confirmed": int(len(confirmed)),
    "classifier_performance_confirmation": classifier_performance_confirmation,
}
with open(STAGE_DIR / "config_used.json", "w") as f:
    json.dump(config_used, f, indent=2, default=str)

# A standalone markdown report -- generated from the actual results above, not hand-written --
# meant to be readable on its own, e.g. dropped into a thesis appendix or shared with a
# supervisor without needing to re-run the notebook.
report_lines = [
    "# MI Acoustic Marker Discovery -- Final Report",
    f"\nGenerated: {config_used['timestamp']}",
    f"\n## Cohorts",
    f"- Development: used for segmentation, training, and candidate discovery (Stages 1-4).",
    f"- Confirmation: {config_used['n_confirmation_patients']} patients, "
    f"{config_used['n_confirmation_recordings']} recordings, untouched until this stage.",
    f"\n## Confirmation-set classifier performance",
    f"- Accuracy: {classifier_performance_confirmation['accuracy']:.3f}",
    f"- Sensitivity: {classifier_performance_confirmation['sensitivity']:.3f}",
    f"- Specificity: {classifier_performance_confirmation['specificity']:.3f}",
    f"- AUC: {classifier_performance_confirmation['auc']:.3f}",
    f"\n## Candidate markers",
    f"{config_used['n_confirmed']} of {config_used['n_candidates_tested']} candidate(s) from "
    f"Stage 4 were confirmed (statistically significant AND direction-consistent) on the "
    f"confirmation cohort.\n",
]
for _, c in confirmation_results.iterrows():
    status = "**CONFIRMED**" if c["confirmed"] else "not confirmed"
    report_lines.append(f"### {c['phase_label']} + {c['freq_low']:.0f}-{c['freq_high']:.0f} Hz + {c['site']} -- {status}")
    if not c["insufficient_data"]:
        direction = "higher" if c["dev_median_mi"] > c["dev_median_normal"] else "lower"
        report_lines.append(
            f"- MI energy {direction} than Normal in development "
            f"(p={c['dev_p_value']:.4g}) and confirmation (p={c['confirm_p_value']:.4g}, "
            f"q={c['confirm_fdr_q_value']:.4g})."
        )
        report_lines.append(f"- Direction agreement: {c['direction_agrees']}")
    else:
        report_lines.append("- Confirmation set had insufficient patients in this cell -- inconclusive.")
    report_lines.append("")

with open(STAGE_DIR / "final_report.md", "w") as f:
    f.write("\n".join(report_lines))

print("Saved:")
for fname in ["phase_boundaries_confirmation.csv", "segmentation_quality_confirmation.csv",
             "confirmation_predictions.csv", "gradcam_heatmaps_confirmation.npz",
             "recording_energy_table_confirmation.csv", "patient_energy_table_confirmation.csv",
             "confirmation_results.csv", "config_used.json", "final_report.md"]:
    print(f"  {STAGE_DIR / fname}")

Saved:
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation/phase_boundaries_confirmation.csv
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation/segmentation_quality_confirmation.csv
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation/confirmation_predictions.csv
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation/gradcam_heatmaps_confirmation.npz
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation/recording_energy_table_confirmation.csv
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation/patient_energy_table_confirmation.csv
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation/confirmation_results.csv
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifacts/05_confirmation/config_used.json
  /content/drive/MyDrive/Research 2026/mi_marker_pipeline/artifact

## Pipeline complete

Five stages, one locked cohort touched exactly once: this notebook's `final_report.md` and
`confirmation_results.csv` are the end product the whole pipeline was built to produce --
whatever candidates confirmed here are the closest thing this project has to an answer for
*"what actually changes in the heart sound when someone has MI, and where."*

If nothing confirmed: that's real information too, and worth reporting as honestly as a
positive result would be -- either the development-set candidates don't hold up, or the
confirmation cohort is too small to tell (worth stating which, given the patient counts above).